# 05 — Model 1b: Cross-Attention + LoRA-only (controlled ablation)

Model 1 (`exp04`, see `notebooks/04_model1_crossattn_training.ipynb`) changed three things at once vs. Stage C -- cross-attention fusion, combined unfreeze(2)+LoRA, and full-res-only data (53,544 rows) -- and **regressed** below Stage C (0.625/0.161 vs. 0.659/0.184), with clear overfitting signs (train loss cratered while val fell, best epoch was only epoch 2).

This isolates ONE variable via `configs/exp05_model1b_crossattn_lora_only.yaml`: **same capacity as Stage C** (LoRA only, no extra unfreeze -- confirmed 0 non-LoRA backbone tensors unfrozen) and the **full dataset** (87,334 rows, not filtered). Only the fusion head changed, concat -> cross-attention (4.0M trainable params vs. Stage C's 986K -- the difference is just the larger head).

**If this beats Stage C (0.659/0.184): cross-attention itself is a real win.**
**If not: the fusion idea may not be it, independent of the overfitting confound.**

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt


## 1. Run training

**Do not run this at the same time as any other GPU/MPS process** -- running two MPS-heavy processes concurrently has caused real hangs (twice now) needing a full machine restart. Run one at a time.

In [ ]:
!cd .. && python3 -u -m src.training.train --config configs/exp05_model1b_crossattn_lora_only.yaml


## 2. Load the logged curves and plot

In [ ]:
run_dir = "../experiments/siglip2_model1b_crossattn_lora_only"

train_log = pd.read_csv(f"{run_dir}/train_log.csv")
val_log = pd.read_csv(f"{run_dir}/val_log.csv")

print("Training steps logged:", len(train_log))
print("Validation epochs logged:", len(val_log))
val_log


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_log["step"], train_log["loss"])
axes[0].set_xlabel("step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Model 1b training loss")

axes[1].plot(val_log["epoch"], val_log["pr_auc"], marker="o", label="PR-AUC")
axes[1].plot(val_log["epoch"], val_log["roc_auc"], marker="o", label="ROC-AUC")
axes[1].axhline(0.212, color="gray", linestyle="--", label="text_only_bert PR-AUC (0.212)")
axes[1].axhline(0.184, color="green", linestyle=":", label="Stage C PR-AUC (0.184)")
axes[1].axhline(0.161, color="lightgray", linestyle=":", label="Model 1 PR-AUC (0.161)")
axes[1].set_xlabel("epoch")
axes[1].set_title("Model 1b validation metrics")
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Compare against every other model tried

In [ ]:
best_1b = val_log.loc[val_log["pr_auc"].idxmax()]

results = pd.DataFrame([
    {"model": "tfidf_logreg",        "roc_auc": 0.660, "pr_auc": 0.208},
    {"model": "text_only_bert",       "roc_auc": 0.704, "pr_auc": 0.212},
    {"model": "image_only",           "roc_auc": 0.619, "pr_auc": 0.148},
    {"model": "title_image_frozen",   "roc_auc": 0.619, "pr_auc": 0.147},
    {"model": "siglip2_stage_a",      "roc_auc": 0.651, "pr_auc": 0.167},
    {"model": "siglip2_stage_c_lora", "roc_auc": 0.659, "pr_auc": 0.184},
    {"model": "model1_crossattn_full","roc_auc": 0.625, "pr_auc": 0.161},
    {"model": "model1b_crossattn_lora_only", "roc_auc": best_1b["roc_auc"], "pr_auc": best_1b["pr_auc"]},
]).set_index("model")

results.sort_values("pr_auc", ascending=False)
